# DINOv2-Enhanced Pix2Vox++
### 6.S058 Final Project — Ishanvi Kommula & Tina Li


---
## Section 0: Numpy Patch + GPU Check

In [ ]:
import numpy as np
np.bool    = np.bool_
np.int     = np.int_
np.float   = np.float64
np.complex = np.complex128
np.object  = object
np.str     = np.str_
print('NumPy', np.__version__, '— aliases patched.')

import torch
print('CUDA:', torch.cuda.is_available())
print('GPU: ', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — change runtime!')
print('PyTorch:', torch.__version__)

NumPy 2.0.2 — aliases patched.
CUDA: True
GPU:  Tesla T4
PyTorch: 2.10.0+cu128


---
## Section 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/6S058_project'
os.makedirs(f'{PROJECT_DIR}/checkpoints', exist_ok=True)
print('Project dir:', PROJECT_DIR)

Mounted at /content/drive
Project dir: /content/drive/MyDrive/6S058_project


---
## Section 2: Extract Data to Local Disk

Extracts from the .tgz files already in your Drive to local Colab SSD.
Much faster than working directly from Drive.


In [ ]:
import os, shutil

LOCAL_DATA = '/content/data'
os.makedirs(LOCAL_DATA, exist_ok=True)

DRIVE_DATA = '/content/drive/MyDrive/6S058_project/data'

def count_cats(path):
    if not os.path.exists(path):
        return 0
    return len([d for d in os.listdir(path) if d.isdigit()])

# ShapeNetRendering
rendering_local = f'{LOCAL_DATA}/ShapeNetRendering'
if count_cats(rendering_local) < 13:
    print('Extracting ShapeNetRendering to local disk...')
    if os.path.exists(rendering_local):
        shutil.rmtree(rendering_local)
    !tar -xzf {DRIVE_DATA}/ShapeNetRendering.tgz -C {LOCAL_DATA}
    print(f'ShapeNetRendering: {count_cats(rendering_local)}/13 categories.')
else:
    print(f'ShapeNetRendering already extracted ({count_cats(rendering_local)}/13).')

# ShapeNetVox32 — download directly to local if not in Drive
voxel_local = f'{LOCAL_DATA}/ShapeNetVox32'
voxel_tgz_drive = f'{DRIVE_DATA}/ShapeNetVox32.tgz'

if count_cats(voxel_local) < 13:
    if os.path.exists(voxel_tgz_drive):
        print('Extracting ShapeNetVox32 from Drive...')
        !tar -xzf {voxel_tgz_drive} -C {LOCAL_DATA}
    else:
        print('Downloading ShapeNetVox32 (~2GB) to local disk...')
        !wget --show-progress -O /content/ShapeNetVox32.tgz http://cvgl.stanford.edu/data2/ShapeNetVox32.tgz
        !tar -xzf /content/ShapeNetVox32.tgz -C {LOCAL_DATA}
        !rm /content/ShapeNetVox32.tgz
    print(f'ShapeNetVox32: {count_cats(voxel_local)}/13 categories.')
else:
    print(f'ShapeNetVox32 already extracted ({count_cats(voxel_local)}/13).')

assert count_cats(rendering_local) == 13, 'ShapeNetRendering incomplete!'
assert count_cats(voxel_local)     == 13, 'ShapeNetVox32 incomplete!'
print('All data ready on local disk!')

Extracting ShapeNetRendering to local disk...
ShapeNetRendering: 13/13 categories.
--2026-05-05 02:35:29--  http://cvgl.stanford.edu/data2/ShapeNetVox32.tgz
Resolving cvgl.stanford.edu (cvgl.stanford.edu)... 171.64.64.64
Connecting to cvgl.stanford.edu (cvgl.stanford.edu)|171.64.64.64|:80... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://cvgl.stanford.edu/data2/ShapeNetVox32.tgz [following]
--2026-05-05 02:35:30--  https://cvgl.stanford.edu/data2/ShapeNetVox32.tgz
Connecting to cvgl.stanford.edu (cvgl.stanford.edu)|171.64.64.64|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 22218020 (21M) [application/x-gzip]
Saving to: ‘/content/ShapeNetVox32.tgz’

/content/ShapeNetVo 100%[===================>]  21.19M  6.10MB/s    in 3.5s    

2026-05-05 02:35:34 (6.10 MB/s) - ‘/content/ShapeNetVox32.tgz’ saved [22218020/22218020]

ShapeNetVox32: 13/13 categories.
All data ready on local disk!


---
## Section 3: Clone Pix2Vox++ and Install Dependencies

In [ ]:
import os
REPO_DIR = '/content/Pix2Vox'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/hzxie/Pix2Vox {REPO_DIR}
    print('Cloned.')
else:
    print('Repo already present.')

%cd {REPO_DIR}
!pip install -q easydict tensorboardX
print('Dependencies ready.')

Cloning into '/content/Pix2Vox'...
remote: Enumerating objects: 812, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 812 (delta 4), reused 5 (delta 4), pack-reused 805 (from 2)
Receiving objects: 100% (812/812), 1.22 MiB | 17.85 MiB/s, done.
Resolving deltas: 100% (509/509), done.
Cloned.
/content/Pix2Vox
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 8.4 MB/s eta 0:00:00
Dependencies ready.


---
## Section 4: Load Config and Set Paths

In [ ]:
import sys
sys.path.insert(0, REPO_DIR)

from config import cfg

LOCAL_DATA = '/content/data'
cfg.DATASETS.SHAPENET.RENDERING_PATH     = LOCAL_DATA + '/ShapeNetRendering/%s/%s/rendering/%02d.png'
cfg.DATASETS.SHAPENET.VOXEL_PATH         = LOCAL_DATA + '/ShapeNetVox32/%s/%s/model.binvox'
cfg.DATASETS.SHAPENET.TAXONOMY_FILE_PATH = REPO_DIR   + '/datasets/ShapeNet.json'

import os
print('Rendering OK:', os.path.exists(LOCAL_DATA + '/ShapeNetRendering'))
print('Voxel OK:    ', os.path.exists(LOCAL_DATA + '/ShapeNetVox32'))
print('Taxonomy OK: ', os.path.exists(cfg.DATASETS.SHAPENET.TAXONOMY_FILE_PATH))
print('All paths set.')

Rendering OK: True
Voxel OK:     True
Taxonomy OK:  True
All paths set.


---
## Section 5: Verify DINOv2 Loads

In [ ]:
import torch
print('Loading DINOv2 ViT-B/14...')
_dino = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14').cuda().eval()
_x    = torch.randn(2, 3, 224, 224).cuda()
with torch.no_grad():
    _out = _dino.get_intermediate_layers(_x, n=1)[0]
print('Output shape:', _out.shape)
assert _out.shape == (2, 256, 768)
print('DINOv2 OK!')
del _dino, _x, _out
torch.cuda.empty_cache()

Loading DINOv2 ViT-B/14...
Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip


/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vitb14/dinov2_vitb14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vitb14_pretrain.pth


100%|██████████| 330M/330M [00:02<00:00, 162MB/s]


Output shape: torch.Size([2, 256, 768])
DINOv2 OK!


---
## Section 6: Define and Write DINOv2Encoder

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F

class DINOv2Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14')
        for p in self.backbone.parameters():
            p.requires_grad = False
        self.projection = nn.Sequential(
            nn.LayerNorm(768),
            nn.Linear(768, 256),
            nn.GELU(),
        )

    def forward(self, x):
        B = x.shape[0]
        with torch.no_grad():
            tokens = self.backbone.get_intermediate_layers(x, n=1)[0]
        out = self.projection(tokens)
        out = out.permute(0, 2, 1).reshape(B, 256, 16, 16)
        out = F.adaptive_avg_pool2d(out, (8, 8))
        return out

_enc = DINOv2Encoder().cuda()
_x   = torch.randn(2, 3, 224, 224).cuda()
_out = _enc(_x)
assert _out.shape == (2, 256, 8, 8), f'Bad shape: {_out.shape}'
print('DINOv2Encoder output shape:', _out.shape, '— OK!')
del _enc, _x, _out
torch.cuda.empty_cache()

src = '''import torch
import torch.nn as nn
import torch.nn.functional as F

class DINOv2Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = torch.hub.load("facebookresearch/dinov2", "dinov2_vitb14")
        for p in self.backbone.parameters():
            p.requires_grad = False
        self.projection = nn.Sequential(
            nn.LayerNorm(768),
            nn.Linear(768, 256),
            nn.GELU(),
        )

    def forward(self, x):
        B = x.shape[0]
        with torch.no_grad():
            tokens = self.backbone.get_intermediate_layers(x, n=1)[0]
        out = self.projection(tokens)
        out = out.permute(0, 2, 1).reshape(B, 256, 16, 16)
        out = F.adaptive_avg_pool2d(out, (8, 8))
        return out
'''
with open(f'{REPO_DIR}/models/dinov2_encoder.py', 'w') as f:
    f.write(src)
print('Written to models/dinov2_encoder.py')

Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main


DINOv2Encoder output shape: torch.Size([2, 256, 8, 8]) — OK!
Written to models/dinov2_encoder.py


---
## Section 7: Experiment Configuration

**This is the only cell you change between experiments.**
- `USE_DINOV2 = False` → baseline run
- `USE_DINOV2 = True`  → DINOv2 run

In [ ]:
import os, json
import torch, torch.nn as nn
from torch.utils.data import DataLoader

# ══════════════════════════════════════════════
USE_DINOV2 = False   # ← CHANGE THIS between runs
N_EPOCHS   = 50      # 50 is enough for the paper
BATCH_SIZE = 64
LR         = 0.001
# ══════════════════════════════════════════════

RUN_NAME = 'dinov2' if USE_DINOV2 else 'baseline'
CKPT_DIR_DRIVE = f'{PROJECT_DIR}/checkpoints/{RUN_NAME}'
os.makedirs(CKPT_DIR_DRIVE, exist_ok=True)

cfg.TRAIN.NUM_EPOCHES = N_EPOCHS
cfg.TRAIN.BATCH_SIZE  = BATCH_SIZE

print(f'Experiment : {RUN_NAME.upper()}')
print(f'Epochs     : {N_EPOCHS}')
print(f'Batch size : {BATCH_SIZE}')
print(f'Checkpoints: {CKPT_DIR_DRIVE}')

Experiment : BASELINE
Epochs     : 50
Batch size : 64
Checkpoints: /content/drive/MyDrive/6S058_project/checkpoints/baseline


---
## Section 8: Build Model

In [ ]:
from models.encoder import Encoder
from models.decoder import Decoder
from models.refiner import Refiner
from models.merger  import Merger

encoder = Encoder(cfg)
decoder = Decoder(cfg)
refiner = Refiner(cfg)
merger  = Merger(cfg)

if USE_DINOV2:
    from models.dinov2_encoder import DINOv2Encoder
    encoder = DINOv2Encoder()
    print('Encoder: DINOv2 ViT-B/14 (frozen backbone + learned projection)')
else:
    print('Encoder: ResNet-18 (baseline)')

encoder = encoder.cuda()
decoder = decoder.cuda()
refiner = refiner.cuda()
merger  = merger.cuda()

n_trainable = lambda m: sum(p.numel() for p in m.parameters() if p.requires_grad)
print(f'Encoder trainable params: {n_trainable(encoder):,}')
print(f'Decoder trainable params: {n_trainable(decoder):,}')

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_BN_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_BN_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vgg16_bn-6c64b313.pth" to /root/.cache/torch/hub/checkpoints/vgg16_bn-6c64b313.pth


100%|██████████| 528M/528M [00:07<00:00, 76.8MB/s]


Encoder: ResNet-18 (baseline)
Encoder trainable params: 4,853,504
Decoder trainable params: 71,583,064


---
## Section 9: Build Data Loaders

In [ ]:
import utils.data_transforms as T
from utils.data_loaders import ShapeNetDataLoader, DatasetType
import torchvision.transforms.functional as TF
import torch

class RGBAtoRGB:
    """Convert RGBA images to RGB by dropping the alpha channel."""
    def __call__(self, rendering_images):
        return [img[:, :, :3] if img.shape[2] == 4 else img
                for img in rendering_images]

IMG_SIZE = (cfg.CONST.IMG_H, cfg.CONST.IMG_W)

train_transforms = T.Compose([
    RGBAtoRGB(),
    T.RandomBackground(cfg.TRAIN.RANDOM_BG_COLOR_RANGE),
    T.ColorJitter(cfg.TRAIN.BRIGHTNESS, cfg.TRAIN.CONTRAST, cfg.TRAIN.SATURATION),
    T.RandomNoise(cfg.TRAIN.NOISE_STD),
    T.Normalize(mean=cfg.DATASET.MEAN, std=cfg.DATASET.STD),
    T.RandomFlip(),
    T.RandomCrop(IMG_SIZE, IMG_SIZE),
    T.ToTensor(),
])
val_transforms = T.Compose([
    RGBAtoRGB(),
    T.Normalize(mean=cfg.DATASET.MEAN, std=cfg.DATASET.STD),
    T.CenterCrop(IMG_SIZE, IMG_SIZE),
    T.ToTensor(),
])

shapenet_loader = ShapeNetDataLoader(cfg)
train_dataset   = shapenet_loader.get_dataset(DatasetType.TRAIN, n_views_rendering=cfg.CONST.N_VIEWS_RENDERING, transforms=train_transforms)
val_dataset     = shapenet_loader.get_dataset(DatasetType.VAL,   n_views_rendering=cfg.CONST.N_VIEWS_RENDERING, transforms=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

print(f'Train samples: {len(train_dataset):,}')
print(f'Val samples  : {len(val_dataset):,}')

[INFO] 2026-05-05 02:36:23.903209 Collecting files of Taxonomy[ID=02691156, Name=aeroplane]
[INFO] 2026-05-05 02:36:25.131492 Collecting files of Taxonomy[ID=02828884, Name=bench]
[INFO] 2026-05-05 02:36:25.515433 Collecting files of Taxonomy[ID=02933112, Name=cabinet]
[INFO] 2026-05-05 02:36:25.972233 Collecting files of Taxonomy[ID=02958343, Name=car]
[WARN] 2026-05-05 02:36:26.271727 Ignore sample 02958343/f9c1d7748c15499c6f2bd1c4e9adb41 since volume file not exists.
[INFO] 2026-05-05 02:36:27.795773 Collecting files of Taxonomy[ID=03001627, Name=chair]
[INFO] 2026-05-05 02:36:30.177036 Collecting files of Taxonomy[ID=03211117, Name=display]
[INFO] 2026-05-05 02:36:30.430663 Collecting files of Taxonomy[ID=03636649, Name=lamp]
[INFO] 2026-05-05 02:36:30.907071 Collecting files of Taxonomy[ID=03691459, Name=speaker]
[INFO] 2026-05-05 02:36:31.448169 Collecting files of Taxonomy[ID=04090263, Name=rifle]
[INFO] 2026-05-05 02:36:31.986644 Collecting files of Taxonomy[ID=04256520, Name=s

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


---
## Section 10: Optimizer and Loss

In [ ]:
import torch.optim as optim

if USE_DINOV2:
    optimizer = optim.Adam([
        {'params': encoder.projection.parameters(), 'lr': LR},
        {'params': decoder.parameters(),            'lr': LR},
        {'params': refiner.parameters(),            'lr': LR},
        {'params': merger.parameters(),             'lr': LR},
    ])
else:
    optimizer = optim.Adam([
        {'params': encoder.parameters(), 'lr': LR},
        {'params': decoder.parameters(), 'lr': LR},
        {'params': refiner.parameters(), 'lr': LR},
        {'params': merger.parameters(),  'lr': LR},
    ])

scheduler = optim.lr_scheduler.MultiStepLR(optimizer, milestones=[25, 40], gamma=0.1)
bce_loss  = nn.BCELoss()
print('Optimizer and scheduler ready.')

Optimizer and scheduler ready.


---
## Section 11: Training and Evaluation Functions

In [ ]:
def train_one_epoch(encoder, decoder, refiner, merger, loader, optimizer):
    if USE_DINOV2:
        encoder.backbone.eval()
        encoder.projection.train()
    else:
        encoder.train()
    decoder.train(); refiner.train(); merger.train()

    total_loss, total_iou, n = 0.0, 0.0, 0

    for batch in loader:
        taxonomy_ids = batch[0]
        model_ids    = batch[1]
        imgs         = batch[2].cuda()
        voxels       = batch[3].cuda()

        B, N, C, H, W = imgs.shape

        if USE_DINOV2:
            features = encoder(imgs.view(B*N, C, H, W)).view(B, N, 256, 8, 8)
        else:
            features = encoder(imgs)

        raw_feats, gen_voxels = decoder(features)
        gen_voxels = merger(raw_feats, gen_voxels)
        gen_voxels = refiner(gen_voxels)

        loss = bce_loss(gen_voxels, voxels.float())
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        with torch.no_grad():
            pred         = (gen_voxels > 0.5).float()
            intersection = (pred * voxels).sum(dim=(1,2,3))
            union        = ((pred + voxels) > 0).float().sum(dim=(1,2,3))
            iou          = (intersection / (union + 1e-6)).mean().item()

        total_loss += loss.item() * B
        total_iou  += iou * B
        n          += B

    return total_loss / n, total_iou / n


@torch.no_grad()
def evaluate(encoder, decoder, refiner, merger, loader):
    encoder.eval(); decoder.eval(); refiner.eval(); merger.eval()

    cat_iou   = {}
    total_iou = 0.0
    n         = 0

    for batch in loader:
        taxonomy_ids = batch[0]
        model_ids    = batch[1]
        imgs         = batch[2].cuda()
        voxels       = batch[3].cuda()

        B, N, C, H, W = imgs.shape

        if USE_DINOV2:
            features = encoder(imgs.view(B*N, C, H, W)).view(B, N, 256, 8, 8)
        else:
            features = encoder(imgs)

        raw_feats, gen_voxels = decoder(features)
        gen_voxels = merger(raw_feats, gen_voxels)
        gen_voxels = refiner(gen_voxels)

        pred         = (gen_voxels > 0.5).float()
        intersection = (pred * voxels).sum(dim=(1,2,3))
        union        = ((pred + voxels) > 0).float().sum(dim=(1,2,3))
        iou_per      = (intersection / (union + 1e-6)).cpu().tolist()

        for tid, iou_val in zip(taxonomy_ids, iou_per):
            cat_iou.setdefault(tid, []).append(iou_val)
            total_iou += iou_val
            n         += 1

    per_cat = {tid: sum(v)/len(v) for tid, v in cat_iou.items()}
    return total_iou / n, per_cat

print('Functions defined.')

Functions defined.


---
## Section 12: Run Training

Only the best model checkpoint is saved to Drive (fast).
Prints every epoch, validates every 5.

In [ ]:
# ── Rebuild transforms with RGBA fix and num_workers=0 ──
import utils.data_transforms as T
from utils.data_loaders import ShapeNetDataLoader, DatasetType
from torch.utils.data import DataLoader

class RGBAtoRGB:
    def __call__(self, imgs):
        result = []
        for img in imgs:
            if hasattr(img, 'shape') and len(img.shape) == 3 and img.shape[2] == 4:
                img = img[:, :, :3]
            elif hasattr(img, 'mode') and img.mode == 'RGBA':
                import numpy as np
                img = np.array(img)[:, :, :3]
            result.append(img)
        return result

IMG_SIZE = (cfg.CONST.IMG_H, cfg.CONST.IMG_W)

train_transforms = T.Compose([
    RGBAtoRGB(),
    T.RandomBackground(cfg.TRAIN.RANDOM_BG_COLOR_RANGE),
    T.ColorJitter(cfg.TRAIN.BRIGHTNESS, cfg.TRAIN.CONTRAST, cfg.TRAIN.SATURATION),
    T.RandomNoise(cfg.TRAIN.NOISE_STD),
    T.Normalize(mean=cfg.DATASET.MEAN, std=cfg.DATASET.STD),
    T.RandomFlip(),
    T.RandomCrop(IMG_SIZE, IMG_SIZE),
    T.ToTensor(),
])
val_transforms = T.Compose([
    RGBAtoRGB(),
    T.Normalize(mean=cfg.DATASET.MEAN, std=cfg.DATASET.STD),
    T.CenterCrop(IMG_SIZE, IMG_SIZE),
    T.ToTensor(),
])

shapenet_loader = ShapeNetDataLoader(cfg)
train_dataset   = shapenet_loader.get_dataset(DatasetType.TRAIN, n_views_rendering=cfg.CONST.N_VIEWS_RENDERING, transforms=train_transforms)
val_dataset     = shapenet_loader.get_dataset(DatasetType.VAL,   n_views_rendering=cfg.CONST.N_VIEWS_RENDERING, transforms=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=False)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False)
print(f'Train: {len(train_dataset):,} | Val: {len(val_dataset):,}')

# ── Restart training from scratch ──
import json
start_epoch = 0
best_iou    = 0.0
history     = {'train_loss': [], 'train_iou': [], 'val_iou': []}

print(f'Starting {RUN_NAME.upper()} for {N_EPOCHS} epochs.')

for epoch in range(start_epoch, N_EPOCHS):
    train_loss, train_iou = train_one_epoch(
        encoder, decoder, refiner, merger, train_loader, optimizer)
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['train_iou'].append(train_iou)

    if (epoch + 1) % 5 == 0 or epoch == N_EPOCHS - 1:
        val_iou, per_cat = evaluate(encoder, decoder, refiner, merger, val_loader)
        history['val_iou'].append({'epoch': epoch, 'mean': val_iou, 'per_cat': per_cat})
        is_best = val_iou > best_iou
        tag = ' <- best!' if is_best else ''
        print(f'[Epoch {epoch+1:03d}/{N_EPOCHS}] '
              f'Loss: {train_loss:.4f} | Train IoU: {train_iou:.4f} | Val IoU: {val_iou:.4f}{tag}')
        if is_best:
            best_iou = val_iou
            torch.save({
                'epoch': epoch, 'best_iou': best_iou, 'history': history,
                'encoder': encoder.state_dict(), 'decoder': decoder.state_dict(),
                'refiner': refiner.state_dict(), 'merger':  merger.state_dict(),
                'optimizer': optimizer.state_dict(),
            }, f'{CKPT_DIR_DRIVE}/best_model.pth')
            print(f'  Saved best model to Drive.')
    else:
        print(f'[Epoch {epoch+1:03d}/{N_EPOCHS}] '
              f'Loss: {train_loss:.4f} | Train IoU: {train_iou:.4f}')

with open(f'{CKPT_DIR_DRIVE}/history.json', 'w') as f:
    json.dump(history, f, indent=2)

print(f'\nDone! Best val IoU: {best_iou:.4f}')

[INFO] 2026-05-05 02:36:36.842886 Collecting files of Taxonomy[ID=02691156, Name=aeroplane]
[INFO] 2026-05-05 02:36:37.648048 Collecting files of Taxonomy[ID=02828884, Name=bench]
[INFO] 2026-05-05 02:36:38.020995 Collecting files of Taxonomy[ID=02933112, Name=cabinet]
[INFO] 2026-05-05 02:36:38.471484 Collecting files of Taxonomy[ID=02958343, Name=car]
[WARN] 2026-05-05 02:36:38.813051 Ignore sample 02958343/f9c1d7748c15499c6f2bd1c4e9adb41 since volume file not exists.
[INFO] 2026-05-05 02:36:40.531690 Collecting files of Taxonomy[ID=03001627, Name=chair]
[INFO] 2026-05-05 02:36:42.239292 Collecting files of Taxonomy[ID=03211117, Name=display]
[INFO] 2026-05-05 02:36:42.427930 Collecting files of Taxonomy[ID=03636649, Name=lamp]
[INFO] 2026-05-05 02:36:42.811827 Collecting files of Taxonomy[ID=03691459, Name=speaker]
[INFO] 2026-05-05 02:36:43.090691 Collecting files of Taxonomy[ID=04090263, Name=rifle]
[INFO] 2026-05-05 02:36:43.493843 Collecting files of Taxonomy[ID=04256520, Name=s

AssertionError: 

---
## Section 13: Evaluate on Test Set
Run after training finishes.

In [ ]:
CATEGORY_NAMES = {
    '02691156': 'airplane',    '02828884': 'bench',
    '02933112': 'cabinet',     '02958343': 'car',
    '03001627': 'chair',       '03211117': 'display',
    '03636649': 'lamp',        '03691459': 'loudspeaker',
    '04090263': 'rifle',       '04256520': 'sofa',
    '04379243': 'table',       '04401088': 'telephone',
    '04530566': 'watercraft',
}

ckpt = torch.load(f'{CKPT_DIR_DRIVE}/best_model.pth')
encoder.load_state_dict(ckpt['encoder'])
decoder.load_state_dict(ckpt['decoder'])
refiner.load_state_dict(ckpt['refiner'])
merger.load_state_dict(ckpt['merger'])
print(f'Loaded best model (epoch {ckpt["epoch"]+1}, val IoU {ckpt["best_iou"]:.4f})')

test_dataset = shapenet_loader.get_dataset(
    DatasetType.TEST, n_views_rendering=cfg.CONST.N_VIEWS_RENDERING, transforms=val_transforms)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

test_iou, per_cat_iou = evaluate(encoder, decoder, refiner, merger, test_loader)

print(f'\n=== TEST RESULTS: {RUN_NAME.upper()} ===')
print(f'{"Category":<20} {"IoU":>8}')
print('-' * 30)
for tid, iou in sorted(per_cat_iou.items(), key=lambda x: CATEGORY_NAMES.get(x[0], x[0])):
    print(f'{CATEGORY_NAMES.get(tid, tid):<20} {iou:>8.4f}')
print('-' * 30)
print(f'{"Mean":<20} {test_iou:>8.4f}')

results = {
    'mean_iou': test_iou,
    'per_category': {CATEGORY_NAMES.get(k, k): v for k, v in per_cat_iou.items()}
}
with open(f'{CKPT_DIR_DRIVE}/test_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print(f'Saved to {CKPT_DIR_DRIVE}/test_results.json')

---
## Section 14: Plot Results
Run after BOTH baseline and DINOv2 experiments are complete.

In [ ]:
import json, numpy as np, matplotlib.pyplot as plt

with open(f'{PROJECT_DIR}/checkpoints/baseline/test_results.json') as f:
    bl = json.load(f)
with open(f'{PROJECT_DIR}/checkpoints/dinov2/test_results.json') as f:
    dv = json.load(f)

cats   = sorted(bl['per_category'].keys())
bl_iou = [bl['per_category'][c] for c in cats]
dv_iou = [dv['per_category'][c] for c in cats]

x, w = np.arange(len(cats)), 0.35
fig, ax = plt.subplots(figsize=(14, 6))
ax.bar(x - w/2, bl_iou, w, label='Pix2Vox++ (ResNet-18)', color='steelblue', alpha=0.85)
ax.bar(x + w/2, dv_iou, w, label='Pix2Vox++ (DINOv2)',   color='coral',     alpha=0.85)
ax.axhline(bl['mean_iou'], color='steelblue', ls='--', lw=1.2,
           label=f'Baseline mean: {bl["mean_iou"]:.3f}')
ax.axhline(dv['mean_iou'], color='coral',     ls='--', lw=1.2,
           label=f'DINOv2 mean:   {dv["mean_iou"]:.3f}')
ax.set_xticks(x); ax.set_xticklabels(cats, rotation=30, ha='right')
ax.set_ylabel('IoU'); ax.set_ylim(0, 1); ax.legend(); ax.grid(axis='y', alpha=0.3)
ax.set_title('Per-Category IoU: Pix2Vox++ Baseline vs DINOv2 Encoder')
plt.tight_layout()
plt.savefig(f'{PROJECT_DIR}/iou_comparison.png', dpi=150)
plt.show()
print('Saved.')

In [ ]:
with open(f'{PROJECT_DIR}/checkpoints/{RUN_NAME}/history.json') as f:
    hist = json.load(f)

val_epochs   = [v['epoch'] for v in hist['val_iou']]
val_iou_vals = [v['mean']  for v in hist['val_iou']]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(hist['train_loss'], color='steelblue')
axes[0].set_title('Training Loss'); axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('BCE Loss');     axes[0].grid(alpha=0.3)

axes[1].plot(hist['train_iou'], color='steelblue', alpha=0.6, label='Train IoU')
axes[1].plot(val_epochs, val_iou_vals, color='coral', marker='o', label='Val IoU')
axes[1].set_title('IoU over Training'); axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('IoU'); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.suptitle(f'Training Curves — {RUN_NAME.upper()}')
plt.tight_layout()
plt.savefig(f'{PROJECT_DIR}/curves_{RUN_NAME}.png', dpi=150)
plt.show()

---
## Section 15: (Optional) Partial Fine-Tuning
Only if frozen DINOv2 underperforms baseline.

In [ ]:
def make_partial_finetune_encoder(n_blocks=2):
    from models.dinov2_encoder import DINOv2Encoder
    enc   = DINOv2Encoder()
    total = len(enc.backbone.blocks)
    for i, block in enumerate(enc.backbone.blocks):
        if i >= total - n_blocks:
            for p in block.parameters():
                p.requires_grad = True
            print(f'  Unfroze block {i}')
    for p in enc.backbone.norm.parameters():
        p.requires_grad = True
    return enc

encoder_ft = make_partial_finetune_encoder(n_blocks=2).cuda()

optimizer_ft = torch.optim.Adam([
    {'params': [p for p in encoder_ft.backbone.parameters() if p.requires_grad], 'lr': 1e-5},
    {'params': encoder_ft.projection.parameters(), 'lr': LR},
    {'params': decoder.parameters(), 'lr': LR},
    {'params': refiner.parameters(), 'lr': LR},
    {'params': merger.parameters(),  'lr': LR},
])

print(f'Trainable params: {sum(p.numel() for p in encoder_ft.parameters() if p.requires_grad):,}')
print('To use: rerun Sections 12-13 with encoder=encoder_ft, optimizer=optimizer_ft')
print('And set RUN_NAME = "dinov2_finetune" manually before running.')

---
## Quick Reference

| What | How |
|---|---|
| **Run baseline** | `USE_DINOV2 = False` in S7, run S7–S13 |
| **Run DINOv2** | `USE_DINOV2 = True` in S7, run S7–S13 |
| **New session** | Run S0–S4 to restore environment, then resume |
| **Resume training** | Set `RESUME_FROM` in S12 |
| **Compare results** | Run S14 after both experiments done |